In [2]:
import csv
import json

# --- ファイル名の設定 ---
FISH_CSV = "fish_list.csv"             # 魚リスト
COUNT_CSV = "分類収録数順.csv"         # 収録数順のCSV
POP_CSV = "分類人気順.csv"             # 人気順のCSV

FISH_JSON = "fish_master.json"
CATEGORY_JSON = "category_master.json"

print("1. 分類マスタ（category_master.json）の作成...")
category_data = {}
with open(COUNT_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for idx, row in enumerate(reader):
        cat_name = row['分類'].strip()
        category_data[cat_name] = {
            "name": cat_name,
            "count": int(row['収録数']),
            "count_rank": idx,
            "pop_rank": 999
        }

with open(POP_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    for idx, row in enumerate(reader):
        if not row:
            continue
        cat_name = row[0].strip()
        if cat_name in category_data:
            category_data[cat_name]["pop_rank"] = idx

category_list = list(category_data.values())
with open(CATEGORY_JSON, 'w', encoding='utf-8') as f:
    json.dump(category_list, f, ensure_ascii=False, indent=2)
print(f" -> {CATEGORY_JSON} 作成完了。")


print("\n2. 魚マスタ（fish_master.json）の作成...")
print("   ※ヘッダー名に左右されないよう、列の順番（位置）で直接取得します。")

fish_list = []
with open(FISH_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    header = next(reader)  # 1行目のヘッダー（列名）をスキップ
    
    for idx, row in enumerate(reader, 1):
        # 行のデータが足りない場合はスキップ
        if len(row) < 5:
            continue
            
        # 列の並び順（0:名前, 1:英名, 2:画像名, 3:分類, 4:解説）で直接指定して取得
        fish_list.append({
            "id": f"fish_{idx:04d}",
            "name": row[0].strip(),
            "english": row[1].strip(),
            "image": row[2].strip(),
            "category": row[3].strip(),  # ← ここで4番目の列を確実に取得！
            "description": row[4].strip()
        })

with open(FISH_JSON, 'w', encoding='utf-8') as f:
    json.dump(fish_list, f, ensure_ascii=False, indent=2)

print(f" -> {FISH_JSON} 作成完了。（総魚数: {len(fish_list)}）")
print("\nすべてのマスタデータが正常に更新されました！")

1. 分類マスタ（category_master.json）の作成...
 -> category_master.json 作成完了。

2. 魚マスタ（fish_master.json）の作成...
   ※ヘッダー名に左右されないよう、列の順番（位置）で直接取得します。
 -> fish_master.json 作成完了。（総魚数: 1323）

すべてのマスタデータが正常に更新されました！


In [5]:
import csv
import json

# --- ファイル名の設定 ---
FISH_CSV = "fish_list.csv"             
COUNT_CSV = "分類収録数順.csv"         
POP_CSV = "分類人気順.csv"             

FISH_JSON = "fish_master.json"
CATEGORY_JSON = "category_master.json"

print("1. 分類マスタ（category_master.json）を再生成しています...")
category_data = {}
with open(COUNT_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for idx, row in enumerate(reader):
        cat_name = row['分類'].strip()
        category_data[cat_name] = {
            "name": cat_name,
            "count": int(row['収録数']),
            "count_rank": idx,
            "pop_rank": 999
        }

with open(POP_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    for idx, row in enumerate(reader):
        if not row:
            continue
        cat_name = row[0].strip()
        if cat_name in category_data:
            category_data[cat_name]["pop_rank"] = idx

category_list = list(category_data.values())
with open(CATEGORY_JSON, 'w', encoding='utf-8') as f:
    json.dump(category_list, f, ensure_ascii=False, indent=2)


print("\n2. 魚マスタ（fish_master.json）を image2, image3 対応版に上書きしています...")
fish_list = []
with open(FISH_CSV, 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    header = next(reader)  # ヘッダーをスキップ
    
    for idx, row in enumerate(reader, 1):
        if len(row) < 5:
            continue
            
        # image2, image3 が追加されているかチェック（安全対策）
        img2 = row[5].strip() if len(row) > 5 else "欠損"
        img3 = row[6].strip() if len(row) > 6 else "欠損"
        
        fish_list.append({
            "id": f"fish_{idx:04d}",
            "name": row[0].strip(),
            "english": row[1].strip(),
            "image": row[2].strip(),
            "image2": img2,   # 追加
            "image3": img3,   # 追加
            "category": row[3].strip(),
            "description": row[4].strip()
        })

with open(FISH_JSON, 'w', encoding='utf-8') as f:
    json.dump(fish_list, f, ensure_ascii=False, indent=2)

print(f" -> {FISH_JSON} を上書き作成しました。（総魚数: {len(fish_list)}）")
print("\nすべてのJSONデータの同期が完了しました！")

1. 分類マスタ（category_master.json）を再生成しています...

2. 魚マスタ（fish_master.json）を image2, image3 対応版に上書きしています...
 -> fish_master.json を上書き作成しました。（総魚数: 1324）

すべてのJSONデータの同期が完了しました！


In [6]:
import json
import re

JSON_FILE = "fish_master.json"

print("fish_master.json を読み込んでいます...")
try:
    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        fish_list = json.load(f)
except FileNotFoundError:
    print(f"エラー: {JSON_FILE} が見つかりません。正しいディレクトリで実行してください。")
    exit()

updated_count = 0

# 各魚の解説文から「人気：」の後ろにある黒星（★）の数をカウントしてプロパティを追加
for fish in fish_list:
    desc = fish.get("description", "")
    
    # 「人気：」の後に続く★や☆、スペースの塊を検索
    match = re.search(r'人気：([★☆\s]+)', desc)
    
    popularity = 0
    if match:
        stars_str = match.group(1)
        popularity = stars_str.count('★') # 黒星の数だけを純粋にカウント
    
    # 魚のデータに新しいプロパティ「popularity」を追加（見つからない場合は0）
    fish["popularity"] = popularity
    updated_count += 1

# アップデートされたデータをJSONファイルに上書き保存
with open(JSON_FILE, 'w', encoding='utf-8') as f:
    json.dump(fish_list, f, ensure_ascii=False, indent=2)

print(f"\n[完了] {updated_count}件の魚データに 'popularity' プロパティを追加して上書き同期しました！")

fish_master.json を読み込んでいます...

[完了] 1324件の魚データに 'popularity' プロパティを追加して上書き同期しました！
